In [1]:
import pandas as pd
from pathlib import Path

file_path = Path("../output/online_retail_clean_backup.csv")

df = pd.read_csv(
    file_path,
    parse_dates=["InvoiceDate"],
    dtype={
        "Invoice": "string",
        "Customer ID": "string"
    }
)

print("Kích thước:", df.shape)
print("Khách hàng:", df["Customer ID"].nunique())
print("Tổng LineTotal:", round(df["LineTotal"].sum(), 2))

Kích thước: (779425, 9)
Khách hàng: 5878
Tổng LineTotal: 17374804.27


In [6]:
snapshot_date = pd.Timestamp("2011-12-10")

# Tổng hợp dữ liệu theo khách hàng
rfm = (
    df.groupby("Customer ID")
    .agg(
        LastPurchase=("InvoiceDate", "max"),
        Frequency=("Invoice", "nunique"),
        Monetary=("LineTotal", "sum")
    )
    .reset_index()
)

# Recency
rfm["Recency"] = (
    snapshot_date - rfm["LastPurchase"].dt.normalize()
).dt.days

rfm = rfm[
    ["Customer ID", "Recency", "Frequency", "Monetary"]
].copy()

print("Ngày tham chiếu:", snapshot_date.date())
print("Kích thước bảng RFM:", rfm.shape)
print("\n10 khách hàng đầu tiên:")
print(rfm.head(10).to_string(index=False))

Ngày tham chiếu: 2011-12-10
Kích thước bảng RFM: (5878, 4)

10 khách hàng đầu tiên:
Customer ID  Recency  Frequency  Monetary
      12346      326         12  77556.46
      12347        3          8   4921.53
      12348       76          5   2019.40
      12349       19          4   4428.69
      12350      311          1    334.40
      12351      376          1    300.93
      12352       37         10   2849.84
      12353      205          2    406.76
      12354      233          1   1079.40
      12355      215          2    947.61


In [7]:
print("KIỂM TRA RFM")

print("Số khách hàng:", len(rfm))
print("Customer ID thiếu:", rfm["Customer ID"].isna().sum())
print("Customer ID trùng:", rfm["Customer ID"].duplicated().sum())

print("Recency <= 0:", (rfm["Recency"] <= 0).sum())
print("Frequency <= 0:", (rfm["Frequency"] <= 0).sum())
print("Monetary <= 0:", (rfm["Monetary"] <= 0).sum())

print("\nĐối chiếu tổng Monetary:")
print("Từ giao dịch:", round(df["LineTotal"].sum(), 2))
print("Từ RFM:", round(rfm["Monetary"].sum(), 2))

print("\nTổng số hóa đơn duy nhất:")
print(df["Invoice"].nunique())

print("\nTổng Frequency theo khách hàng:")
print(rfm["Frequency"].sum())

KIỂM TRA RFM
Số khách hàng: 5878
Customer ID thiếu: 0
Customer ID trùng: 0
Recency <= 0: 0
Frequency <= 0: 0
Monetary <= 0: 0

Đối chiếu tổng Monetary:
Từ giao dịch: 17374804.27
Từ RFM: 17374804.27

Tổng số hóa đơn duy nhất:
36969

Tổng Frequency theo khách hàng:
36969


In [5]:
rfm_summary = rfm[
    ["Recency", "Frequency", "Monetary"]
].describe()

print("THỐNG KÊ RFM")
print(rfm_summary.round(2).to_string())

THỐNG KÊ RFM
       Recency  Frequency   Monetary
count  5878.00    5878.00    5878.00
mean    201.87       6.29    2955.90
std     209.35      13.01   14440.85
min       1.00       1.00       2.95
25%      26.00       1.00     342.28
50%      96.00       3.00     867.74
75%     380.00       7.00    2248.30
max     739.00     398.00  580987.04


In [8]:
from pathlib import Path

output_dir = Path("../output")
output_dir.mkdir(parents=True, exist_ok=True)

rfm.to_csv(
    output_dir / "customer_rfm.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Done")
print("Đường dẫn:", (output_dir / "customer_rfm.csv").resolve())

Done
Đường dẫn: C:\Users\Admin\Desktop\Học tập\Thực tập & tốt nghiệp\analytics\output\customer_rfm.csv
